In [2]:
from __future__ import annotations

import logging
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from scipy import stats
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from rule_backtest.signals import SIGNAL_CLASSES, BaseSignal
from rule_backtest.data_loader import (
    load_cached_trades,
    merge_aggtrades,
    build_bars,
    compute_factor,
    estimate_avg_bar_seconds,
    RAW_FACTOR_REGISTRY,
    MERGED_FACTOR_REGISTRY,
)

logger = logging.getLogger(__name__)

In [3]:
cache_dir = Path("/Volumes/Lexar/mean_reversion_data/cache")

In [4]:
symbol='SOL/USDC'
cache_subdir: str = "trades_perp"
sym_dir = cache_dir / symbol.replace("/", "_") / cache_subdir
if not sym_dir.exists():
    raise FileNotFoundError(f"Cache directory not found: {sym_dir}")

start_date = '2026-02-02'
end_date = '2026-03-04'
files = sorted(f for f in sym_dir.glob("*.parquet") if not f.name.startswith("._") and start_date <= f.stem <= end_date)

if not files:
    raise FileNotFoundError(f"No parquet files in {sym_dir}")

logger.info("Loading %d cached files from %s", len(files), sym_dir)
dfs = [pd.read_parquet(f) for f in files]
trades = pd.concat(dfs, ignore_index=True)
trades["timestamp"] = pd.to_datetime(trades["timestamp"])
trades.sort_values("timestamp", inplace=True)
trades.reset_index(drop=True, inplace=True)

# Unified column names
trades.rename(columns={"amount": "volume", "cost": "value"}, inplace=True)

logger.info("Loaded %d trades, %s → %s",
            len(trades), trades["timestamp"].iloc[0], trades["timestamp"].iloc[-1])

In [5]:
trades_df = merge_aggtrades(trades)
trades_df.head()

,timestamp,volume,side,value,n_fills,n_raw_trades,first_price,last_price,first_trade_id,last_trade_id,price,price_range
0,2026-02-02 16:00:01.690000+00:00,16.45,sell,1720.3798,2,7,104.59,104.58,304094557,304094563,104.582359,0.01
1,2026-02-02 16:00:01.843000+00:00,40.10,buy,4193.8513,2,3,104.58,104.59,304094564,304094566,104.584820,0.01
2,2026-02-02 16:00:01.970000+00:00,62.43,sell,6528.9294,1,3,104.58,104.58,304094567,304094569,104.580000,0.00
3,2026-02-02 16:00:02.100000+00:00,5.02,sell,525.0418,1,3,104.59,104.59,304094570,304094572,104.590000,0.00
4,2026-02-02 16:00:02.360000+00:00,0.06,buy,6.2754,1,1,104.59,104.59,304094573,304094573,104.590000,0.00


In [14]:
# Bar spec：与 build_bars 签名一致 — 第 4 个位置参数是 freq(str)，不是 tpb
bar_mode = "trade_count"
trades_per_bar = 200
bars = build_bars(trades_df, "merged", bar_mode, freq="1min", trades_per_bar=trades_per_bar)
bar_sec = int(estimate_avg_bar_seconds(bars))
logger.info("Built %d trade-count bars (tpb=%d, ~%ds avg)", len(bars), trades_per_bar, bar_sec)

In [15]:
bars

,open,high,low,close,volume,n_trades,buy_volume,sell_volume,buy_trade_count,sell_trade_count,...,buy_mf_vol_median_ratio,sell_mf_vol_median_ratio,buy_vwap_dist_reg_beta,sell_vwap_dist_reg_beta,buy_pr_reg_beta,sell_pr_reg_beta,buy_volume_skew,sell_volume_skew,weighted_vwap_dist,bar_duration_sec
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-02-02 16:00:01.690,104.582359,104.590000,104.3800,104.470000,5398.11,200,3112.02,2286.09,102,98,...,-0.835453,-1.419416,0.000401,0.001406,0.001324,0.002367,2.731606,4.052468,-8.1451,78.893
2026-02-02 16:01:21.081,104.479890,104.610000,104.4000,104.570000,6154.29,200,3111.91,3042.38,99,101,...,-1.386190,-1.993089,0.000931,0.000774,0.001382,0.001438,3.968450,2.592667,18.7271,94.950
2026-02-02 16:02:56.502,104.574022,104.590000,104.3500,104.410000,9101.02,200,2751.99,6349.03,90,110,...,-0.831131,-1.212047,0.000762,0.000673,0.001464,0.001140,3.025289,4.481283,-7.1579,99.812
2026-02-02 16:04:36.466,104.420000,104.460000,104.1700,104.460000,9769.26,200,4654.26,5115.00,90,110,...,-0.849369,-1.705203,0.000965,0.001114,0.002026,0.001963,7.378304,7.604990,-60.9893,79.668
2026-02-02 16:05:56.828,104.470000,104.740000,104.4300,104.740000,5534.80,200,2683.71,2851.09,89,111,...,-0.459037,-2.049292,0.000756,0.000547,0.001226,0.000997,2.824415,2.596306,4.5561,80.414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-04 15:55:42.573,92.479479,92.610000,92.3200,92.590000,6497.63,200,3438.13,3059.50,113,87,...,-0.419143,-1.953313,0.000312,0.000888,0.000459,0.001478,2.986988,4.876661,-9.4826,115.505
2026-03-04 15:57:39.695,92.582723,92.982745,92.5500,92.799964,15382.10,200,10561.17,4820.93,110,90,...,-1.797607,-1.465275,0.002118,0.000738,0.004233,0.001727,7.045531,3.618333,175.8797,29.512
2026-03-04 15:58:09.265,92.800000,93.080000,92.8000,92.980000,18363.20,200,9107.97,9255.23,103,97,...,-0.985031,-0.683010,0.001396,0.001307,0.002676,0.002206,7.258915,7.205944,9.7580,22.944


## helper functions

In [7]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from datetime import datetime

def calculate_trading_metrics(bars, position_col, returns_col, fee_rate, factor_name, fw, ema_w):
    """
    计算全面的交易绩效指标（修正版：正确处理反向交易）
    """
    
    # 基础数据准备
    df = bars.copy()
    
    # 计算原始收益（不含费用）
    df['raw_returns'] = df[position_col].shift(1) * df['returns']
    
    # 识别持仓变化
    df['position_prev'] = df[position_col].shift(1).fillna(0)
    df['position_change'] = df[position_col] - df['position_prev']
    
    # 初始化信号列
    df['open_signal'] = False
    df['close_signal'] = False
    
    # 识别开仓和平仓时刻（正确处理反向交易）
    for i in range(1, len(df)):
        change = df['position_change'].iloc[i]
        prev_pos = df['position_prev'].iloc[i]
        curr_pos = df[position_col].iloc[i]
        
        if change == 0:
            continue
        
        if abs(change) == 1:
            # 普通开仓或平仓
            if prev_pos == 0:  # 0 -> ±1，开仓
                df.loc[df.index[i], 'open_signal'] = True
            elif curr_pos == 0:  # ±1 -> 0，平仓
                df.loc[df.index[i], 'close_signal'] = True
                
        elif abs(change) == 2:
            # 反向交易：先平仓，再开仓
            # 平仓发生在 bar i 开盘（使用上一根 bar 的持仓）
            df.loc[df.index[i], 'close_signal'] = True
            # 开仓也发生在 bar i 开盘（新方向）
            df.loc[df.index[i], 'open_signal'] = True
    
    # 构建交易列表
    trades = []
    current_trade = None
    
    for i in range(len(df)):
        # 先处理平仓（如果有）
        if df['close_signal'].iloc[i] and current_trade is not None:
            # 平仓
            current_trade['exit_time'] = df.index[i]
            current_trade['exit_price'] = df['open'].iloc[i]  # 以开盘价平仓
            trades.append(current_trade)
            current_trade = None
        
        # 再处理开仓
        if df['open_signal'].iloc[i]:
            direction = df[position_col].iloc[i]  # 当前持仓方向
            current_trade = {
                'entry_time': df.index[i],
                'entry_price': df['open'].iloc[i],
                'direction': direction,
                'exit_time': None,
                'exit_price': None,
                'return_raw': 0,
                'bars_held': 0
            }
        
        # 更新当前持仓的收益
        if current_trade is not None:
            current_trade['return_raw'] += df['raw_returns'].iloc[i]
            current_trade['bars_held'] += 1
    
    # 处理最后一笔未平仓交易
    if current_trade is not None:
        trades.append(current_trade)
    
    # 转换为DataFrame
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame()
    
    # 统计指标
    if not trades_df.empty:
        # 基本交易统计
        n_trades_total = len(trades_df)
        long_trades = trades_df[trades_df['direction'] == 1]
        short_trades = trades_df[trades_df['direction'] == -1]
        n_long = len(long_trades)
        n_short = len(short_trades)
        
        # 胜率计算
        win_rate_total = (trades_df['return_raw'] > 0).mean() * 100
        win_rate_long = (long_trades['return_raw'] > 0).mean() * 100 if n_long > 0 else 0
        win_rate_short = (short_trades['return_raw'] > 0).mean() * 100 if n_short > 0 else 0
        
        # 盈亏比相关
        winning_trades = trades_df[trades_df['return_raw'] > 0]
        losing_trades = trades_df[trades_df['return_raw'] < 0]
        
        avg_win = winning_trades['return_raw'].mean() if len(winning_trades) > 0 else 0
        avg_loss = losing_trades['return_raw'].mean() if len(losing_trades) > 0 else 0
        
        profit_factor = abs(avg_win * len(winning_trades) / (avg_loss * len(losing_trades))) if len(losing_trades) > 0 and avg_loss != 0 else float('inf')
        
        # 最大连续亏损
        trades_df['is_win'] = trades_df['return_raw'] > 0
        trades_df['consecutive_losses'] = (~trades_df['is_win']).groupby((trades_df['is_win'] != trades_df['is_win'].shift()).cumsum()).cumsum()
        max_consecutive_losses = trades_df['consecutive_losses'].max()
        
        # 平均持仓bar数
        avg_bars_held = trades_df['bars_held'].mean()
        
        # 最佳/最差交易
        best_trade = trades_df['return_raw'].max()
        worst_trade = trades_df['return_raw'].min()
        expectancy = trades_df['return_raw'].mean()
        
    else:
        n_trades_total = n_long = n_short = 0
        win_rate_total = win_rate_long = win_rate_short = 0
        profit_factor = 0
        max_consecutive_losses = 0
        avg_bars_held = 0
        best_trade = worst_trade = expectancy = 0
        avg_win = 0  # 添加这一行
        avg_loss = 0  # 添加这一行
        long_trades = short_trades = pd.DataFrame()
    
    # 累计收益（不含费用）
    df['cumulative_raw'] = (1 + df['raw_returns']).cumprod()
    df['cumulative_long_raw'] = (1 + df['raw_returns'] * (df[position_col] == 1).astype(int)).cumprod()
    df['cumulative_short_raw'] = (1 + df['raw_returns'] * (df[position_col] == -1).astype(int)).cumprod()
    
    # 累计收益（含费用）
    df['cumulative_net'] = (1 + df[returns_col]).cumprod()
    
    # 风险指标
    risk_free_rate = 0  # 0%年化无风险利率，分钟级数据
    
    # 夏普比率
    excess_returns = df[returns_col] - risk_free_rate
    sharpe_ratio = np.sqrt(len(df)) * excess_returns.mean() / excess_returns.std() if excess_returns.std() > 0 else 0
    
    # 最大回撤
    rolling_max = df['cumulative_net'].expanding().max()
    drawdown = (df['cumulative_net'] - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    max_drawdown_duration = (drawdown == 0).astype(int).groupby(drawdown.ne(0).astype(int).cumsum()).cumsum().max()
    
    # 卡尔玛比率
    calmar_ratio = (df['cumulative_net'].iloc[-1] - 1) / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # 费用统计
    total_fees = (fee_rate * df['position_change'].abs()).sum()
    avg_fee_per_trade = total_fees / n_trades_total if n_trades_total > 0 else 0
    
    # 反向交易统计（用于诊断）
    n_reversals = (df['position_change'].abs() == 2).sum()
    
    # 年化指标（假设分钟级数据）
    bars_per_year = 365 * 24 * 60
    total_return_net = df['cumulative_net'].iloc[-1] - 1
    annualized_return = (1 + total_return_net) ** (bars_per_year / len(df)) - 1
    annualized_volatility = df[returns_col].std() * np.sqrt(bars_per_year)
    
    # 收集所有指标
    metrics = {
        # 基本信息
        'factor_name': factor_name,
        'fw': fw,
        'ema_w': ema_w,
        'start_date': df.index[0],
        'end_date': df.index[-1],
        'total_bars': len(df),
        
        # 收益指标
        'total_return_net': total_return_net,
        'total_return_long_raw': df['cumulative_long_raw'].iloc[-1] - 1,
        'total_return_short_raw': df['cumulative_short_raw'].iloc[-1] - 1,
        'benchmark_return': df['benchmark_cumulative'].iloc[-1] - 1,
        'annualized_return': annualized_return,
        'annualized_volatility': annualized_volatility,
        
        # 交易统计
        'n_trades_total': n_trades_total,
        'n_long': n_long,
        'n_short': n_short,
        'n_reversals': n_reversals,
        'avg_bars_held': avg_bars_held,
        
        # 胜率
        'win_rate_total': win_rate_total,
        'win_rate_long': win_rate_long,
        'win_rate_short': win_rate_short,
        
        # 盈亏分析
        'profit_factor': profit_factor,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'expectancy': expectancy,
        'best_trade': best_trade,
        'worst_trade': worst_trade,
        'max_consecutive_losses': max_consecutive_losses,
        
        # 风险指标
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'max_drawdown_duration': max_drawdown_duration,
        'calmar_ratio': calmar_ratio,
        
        # 费用统计
        'total_fees': total_fees,
        'avg_fee_per_trade': avg_fee_per_trade,
        'fee_rate': fee_rate,
    }
    
    return metrics, trades_df, df


def print_metrics_summary(metrics):
    """打印指标汇总"""
    print("=" * 60)
    print(f"策略绩效报告 - {metrics['factor_name']} (fw={metrics['fw']}, ema={metrics['ema_w']})")
    print("=" * 60)
    print(f"回测期间: {metrics['start_date']} 至 {metrics['end_date']}")
    print(f"总Bar数: {metrics['total_bars']}")
    print("-" * 60)
    print("【收益指标】")
    print(f"  策略总收益 (净): {metrics['total_return_net']:.2%}")
    print(f"  多头收益 (原始): {metrics['total_return_long_raw']:.2%}")
    print(f"  空头收益 (原始): {metrics['total_return_short_raw']:.2%}")
    print(f"  基准收益: {metrics['benchmark_return']:.2%}")
    print(f"  年化收益率: {metrics['annualized_return']:.2%}")
    print("-" * 60)
    print("【交易统计】")
    print(f"  总交易次数: {metrics['n_trades_total']}")
    print(f"  多头交易: {metrics['n_long']}")
    print(f"  空头交易: {metrics['n_short']}")
    print(f"  反向交易: {metrics['n_reversals']}")
    print(f"  平均持仓Bar数: {metrics['avg_bars_held']:.2f}")
    print("-" * 60)
    print("【胜率统计】")
    print(f"  总胜率: {metrics['win_rate_total']:.2f}%")
    print(f"  多头胜率: {metrics['win_rate_long']:.2f}%")
    print(f"  空头胜率: {metrics['win_rate_short']:.2f}%")
    print("-" * 60)
    print("【盈亏分析】")
    print(f"  盈亏比 (Profit Factor): {metrics['profit_factor']:.3f}")
    print(f"  平均盈利: {metrics['avg_win']:.4%}")
    print(f"  平均亏损: {metrics['avg_loss']:.4%}")
    print(f"  单笔期望收益: {metrics['expectancy']:.4%}")
    print(f"  最佳交易: {metrics['best_trade']:.4%}")
    print(f"  最差交易: {metrics['worst_trade']:.4%}")
    print("-" * 60)
    print("【风险指标】")
    print(f"  夏普比率: {metrics['sharpe_ratio']:.3f}")
    print(f"  最大回撤: {metrics['max_drawdown']:.2%}")
    print(f"  最大回撤持续期 (bars): {metrics['max_drawdown_duration']}")
    print(f"  卡尔玛比率: {metrics['calmar_ratio']:.3f}")
    print(f"  年化波动率: {metrics['annualized_volatility']:.2%}")
    print("-" * 60)
    print("【费用统计】")
    print(f"  总手续费: {metrics['total_fees']:.6f}")
    print(f"  平均每笔手续费: {metrics['avg_fee_per_trade']:.6f}")
    print(f"  费率设置: {metrics['fee_rate']:.6f}")
    print("=" * 60)


def save_metrics_to_file(metrics, trades_df, output_dir, factor_name, fw, ema_w):
    """保存指标到文件"""
    # 创建输出目录
    save_dir = output_dir / factor_name / f"fw{fw}_ema{ema_w}" / "metrics"
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # 保存指标摘要为JSON
    metrics_serializable = {k: (str(v) if isinstance(v, pd.Timestamp) else v) 
                           for k, v in metrics.items()}
    with open(save_dir / "metrics_summary.json", 'w') as f:
        json.dump(metrics_serializable, f, indent=4, default=str)
    
    # 保存交易明细为CSV
    if not trades_df.empty:
        trades_df.to_csv(save_dir / "trades_detail.csv")
    
    # 保存文本报告
    with open(save_dir / "report.txt", 'w') as f:
        # 重定向print输出到文件
        import sys
        original_stdout = sys.stdout
        sys.stdout = f
        print_metrics_summary(metrics)
        sys.stdout = original_stdout


def plot_enhanced_results(bars, metrics, trades_df, factor_name, fw, ema_w, output_dir):
    """绘制增强版结果图表"""
    
    # 创建3行图表
    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=True,
        subplot_titles=('累计收益对比', '策略 vs 基准（滚动）', 
                       '多空累计收益（不含费用）', '回撤曲线',
                       '交易分布', '月度收益热力图'),
        vertical_spacing=0.08,
        horizontal_spacing=0.1
    )
    
    # 1. 累计收益对比（含费用）
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars['benchmark_cumulative'], 
                  mode='lines', name='基准', line=dict(color='#1f77b4', width=1)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars[f'strategy_cumulative_{factor_name}'], 
                  mode='lines', name='策略（净）', line=dict(color='#2ecc71', width=2)),
        row=1, col=1
    )
    
    # 2. 滚动夏普比率（使用60期窗口）
    rolling_sharpe = bars[f'strategy_returns_{factor_name}'].rolling(60).mean() / \
                     bars[f'strategy_returns_{factor_name}'].rolling(60).std() * np.sqrt(60)
    fig.add_trace(
        go.Scatter(x=bars.index, y=rolling_sharpe, 
                  mode='lines', name='滚动夏普(60期)', 
                  line=dict(color='#ff7f0e', width=1)),
        row=1, col=2
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=2)
    
    # 3. 多空累计收益（不含费用）
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars['cumulative_long_raw'], 
                  mode='lines', name='多头收益（原始）', line=dict(color='#d62728', width=1)),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=bars.index, y=bars['cumulative_short_raw'], 
                  mode='lines', name='空头收益（原始）', line=dict(color='#9467bd', width=1)),
        row=2, col=1
    )
    
    # 4. 回撤曲线
    rolling_max = bars[f'strategy_cumulative_{factor_name}'].expanding().max()
    drawdown = (bars[f'strategy_cumulative_{factor_name}'] - rolling_max) / rolling_max
    fig.add_trace(
        go.Scatter(x=bars.index, y=drawdown, fill='tozeroy',
                  mode='lines', name='回撤', line=dict(color='#e377c2', width=1)),
        row=2, col=2
    )
    fig.add_hline(y=metrics['max_drawdown'], line_dash="dash", 
                  line_color="red", opacity=0.5, row=2, col=2)
    
    # 5. 交易分布散点图
    if not trades_df.empty:
        # 多头交易
        long_trades = trades_df[trades_df['direction'] == 1]
        short_trades = trades_df[trades_df['direction'] == -1]
        
        fig.add_trace(
            go.Scatter(x=long_trades['entry_time'] if not long_trades.empty else [],
                      y=long_trades['return_raw'] if not long_trades.empty else [],
                      mode='markers', name='多头交易',
                      marker=dict(color='green', size=8, symbol='triangle-up'),
                      text=[f"收益: {r:.2%}" for r in long_trades['return_raw']] if not long_trades.empty else []),
            row=3, col=1
        )
        fig.add_trace(
            go.Scatter(x=short_trades['entry_time'] if not short_trades.empty else [],
                      y=short_trades['return_raw'] if not short_trades.empty else [],
                      mode='markers', name='空头交易',
                      marker=dict(color='red', size=8, symbol='triangle-down'),
                      text=[f"收益: {r:.2%}" for r in short_trades['return_raw']] if not short_trades.empty else []),
            row=3, col=1
        )
    
    # 6. 收益分布直方图
    fig.add_trace(
        go.Histogram(x=bars[f'strategy_returns_{factor_name}'], nbinsx=50,
                    name='收益分布', marker_color='#7f7f7f'),
        row=3, col=2
    )
    fig.add_vline(x=0, line_dash="dash", line_color="red", row=3, col=2)
    
    # 更新布局
    fig.update_layout(
        height=1200,
        showlegend=True,
        title_text=f"策略分析报告 - {factor_name} (fw={fw}, ema={ema_w})",
        template="plotly_white",
        hovermode='x unified'
    )
    
    # 更新坐标轴标签
    fig.update_xaxes(title_text="时间", row=3, col=1)
    fig.update_xaxes(title_text="时间", row=3, col=2)
    fig.update_yaxes(title_text="累计收益", row=1, col=1)
    fig.update_yaxes(title_text="夏普比率", row=1, col=2)
    fig.update_yaxes(title_text="累计收益", row=2, col=1)
    fig.update_yaxes(title_text="回撤", row=2, col=2, tickformat='.1%')
    fig.update_yaxes(title_text="交易收益", row=3, col=1, tickformat='.1%')
    fig.update_yaxes(title_text="频次", row=3, col=2)
    
    # 保存
    html_path = output_dir / factor_name / f"fw{fw}_ema{ema_w}" / "enhanced_analysis.html"
    fig.write_html(str(html_path), include_plotlyjs="cdn")
    
    return fig

## SOTA

In [16]:
def calculate_delta_threshold(delta_series, window=100, method='mad', k=1, vol_adjusted=False, vol_series=None, local_window=10):
    """
    计算delta的自适应阈值

    Parameters:
    -----------
    delta_series : pd.Series
        delta值序列
    window : int
        滚动窗口
    method : str
        'mad' - 基于绝对中位差（推荐，稳健）
        'std' - 基于标准差
        'quantile' - 基于分位数
    k : float
        阈值倍数
    """
    if method == 'mad':
        rolling_median = delta_series.rolling(window).median()
        abs_dev = (delta_series - rolling_median).abs()
        rolling_mad = abs_dev.rolling(window).median()
        threshold = rolling_mad * k
    elif method == 'std':
        rolling_std = delta_series.rolling(window).std()
        threshold = rolling_std * k
    elif method == 'quantile':
        threshold = delta_series.rolling(window).quantile(0.75)

    if vol_adjusted:
        local_vol_series = vol_series.rolling(local_window).mean()
        avg_vol = local_vol_series.rolling(window).mean()
        threshold = threshold * np.sqrt(local_vol_series / avg_vol)

    return threshold.fillna(method='bfill').fillna(method='ffill')


def vwap_dist_sum_v41_strategy_enhanced(bars, fw1, ema_w1, fw2, ema_w2, fill_rate=0.7, save_metrics=True, plot_enhanced=True):
    """
    增强版策略函数，包含全面绩效指标记录
    """
    factor_name = 'vwap_dist_sum_imbalance_v41'

    bars[f'buy_vwap_dist_sum_fw{fw1}'] = bars['buy_vwap_dist_sum'].rolling(fw1).mean()
    bars[f'sell_vwap_dist_sum_fw{fw1}'] = bars['sell_vwap_dist_sum'].rolling(fw1).mean()
    bars[f'{factor_name}_fw{fw1}'] = (bars[f'buy_vwap_dist_sum_fw{fw1}'] - bars[f'sell_vwap_dist_sum_fw{fw1}']) / (bars[f'buy_vwap_dist_sum_fw{fw1}'] + bars[f'sell_vwap_dist_sum_fw{fw1}'] + 1e-10)
    bars[f'{factor_name}_fw{fw1}_ema{ema_w1}'] = bars[f'{factor_name}_fw{fw1}'].ewm(span=ema_w1, min_periods=1).mean()

    bars[f'buy_vwap_dist_sum_fw{fw2}'] = bars['buy_vwap_dist_sum'].rolling(fw2).mean()
    bars[f'sell_vwap_dist_sum_fw{fw2}'] = bars['sell_vwap_dist_sum'].rolling(fw2).mean()
    bars[f'{factor_name}_fw{fw2}'] = (bars[f'buy_vwap_dist_sum_fw{fw2}'] - bars[f'sell_vwap_dist_sum_fw{fw2}']) / (bars[f'buy_vwap_dist_sum_fw{fw2}'] + bars[f'sell_vwap_dist_sum_fw{fw2}'] + 1e-10)
    bars[f'{factor_name}_fw{fw2}_ema{ema_w2}'] = bars[f'{factor_name}_fw{fw2}'].ewm(span=ema_w2, min_periods=1).mean()

    bars[f'delta1'] = bars[f'{factor_name}_fw{fw1}_ema{ema_w1}'].diff()
    bars[f'delta2'] = bars[f'{factor_name}_fw{fw2}_ema{ema_w2}'].diff()

    bars['delta_threshold1'] = calculate_delta_threshold(
        bars['delta1'],
        window=100,
        method='mad',
        k=1.5,
        vol_adjusted=True, vol_series=bars['volume'], local_window=10
    )
    bars['delta_threshold2'] = calculate_delta_threshold(
        bars['delta2'],
        window=100,
        method='mad',
        k=1,
        vol_adjusted=True, vol_series=bars['volume'], local_window=10
    )

    bars['delta_sign_raw1'] = np.sign(bars['delta1'])
    bars['delta_sign1'] = bars['delta_sign_raw1'].replace(0, method='ffill')
    bars['delta_sign_raw2'] = np.sign(bars['delta2'])
    bars['delta_sign2'] = bars['delta_sign_raw2'].replace(0, method='ffill')

    bars['reversal_neg'] = (bars['delta_sign2'].shift(1) == 1) & (bars['delta_sign2'] == -1)
    bars['reversal_pos'] = (bars['delta_sign1'].shift(1) == -1) & (bars['delta_sign1'] == 1)

    filter_delta1 = bars['delta1'].abs() > bars['delta_threshold1']
    filter_delta2 = bars['delta2'].abs() > bars['delta_threshold2']

    bars[f'short_entry_{factor_name}'] = bars['reversal_neg'] & filter_delta2
    bars[f'long_entry_{factor_name}'] = bars['reversal_pos'] & filter_delta1

    bars[f'position_{factor_name}'] = np.nan
    bars.loc[bars[f'short_entry_{factor_name}'], f'position_{factor_name}'] = -1
    bars.loc[bars[f'long_entry_{factor_name}'], f'position_{factor_name}'] = 1
    bars[f'position_{factor_name}'].ffill(inplace=True)
    bars[f'position_{factor_name}'] = bars[f'position_{factor_name}'].fillna(0)

    fee_rate = (2 * fill_rate + 5 * (1 - fill_rate)) * 1e-4

    bars['returns'] = (bars['close'] - bars['open']) / bars['open']
    bars[f'strategy_returns_{factor_name}'] = bars[f'position_{factor_name}'].shift(1) * bars['returns']

    bars['position_change'] = bars[f'position_{factor_name}'].diff().fillna(0)
    bars[f'strategy_returns_{factor_name}'] = bars[f'strategy_returns_{factor_name}'] - fee_rate * bars['position_change'].abs()

    bars['benchmark_cumulative'] = (1 + bars['returns']).cumprod()
    bars[f'strategy_cumulative_{factor_name}'] = (1 + bars[f'strategy_returns_{factor_name}']).cumprod()

    metrics, trades_df, bars_with_metrics = calculate_trading_metrics(
        bars, f'position_{factor_name}', f'strategy_returns_{factor_name}',
        fee_rate, factor_name, fw1, ema_w1
    )

    print_metrics_summary(metrics)

    output_dir = Path("/Volumes/Lexar/mean_reversion_data/backtest_results")
    if save_metrics:
        save_metrics_to_file(metrics, trades_df, output_dir, factor_name, fw1, ema_w1)

    if plot_enhanced:
        plot_enhanced_results(bars_with_metrics, metrics, trades_df, factor_name, fw1, ema_w1, output_dir)

    return bars_with_metrics, metrics, trades_df

#### （可选）运行 SOTA 基线

在 **`bars.copy()`** 上调用 `vwap_dist_sum_v41_strategy_enhanced`，避免改写下方共用的 `bars`。
若不需要对照基线，可跳过本节。

In [17]:
_sota_bars, _m, _t = vwap_dist_sum_v41_strategy_enhanced(
    bars.copy(), 8, 10, 10, 10, save_metrics=True, plot_enhanced=True
)

策略绩效报告 - vwap_dist_sum_imbalance_v41 (fw=8, ema=10)
回测期间: 2026-02-02 16:00:01.690000 至 2026-03-04 15:59:09.639000
总Bar数: 15163
------------------------------------------------------------
【收益指标】
  策略总收益 (净): 66.32%
  多头收益 (原始): 26.27%
  空头收益 (原始): 38.92%
  基准收益: -11.70%
  年化收益率: 4560411327.17%
------------------------------------------------------------
【交易统计】
  总交易次数: 92
  多头交易: 46
  空头交易: 46
  反向交易: 91
  平均持仓Bar数: 162.62
------------------------------------------------------------
【胜率统计】
  总胜率: 54.35%
  多头胜率: 56.52%
  空头胜率: 52.17%
------------------------------------------------------------
【盈亏分析】
  盈亏比 (Profit Factor): 1.958
  平均盈利: 2.4646%
  平均亏损: -1.4989%
  单笔期望收益: 0.6552%
  最佳交易: 9.8929%
  最差交易: -10.5874%
------------------------------------------------------------
【风险指标】
  夏普比率: 1.921
  最大回撤: -27.10%
  最大回撤持续期 (bars): 201
  卡尔玛比率: 2.448
  年化波动率: 168.51%
------------------------------------------------------------
【费用统计】
  总手续费: 0.053070
  平均每笔手续费: 0.000577
  费率设置: 0.000290


### 研究分支：`v41_strategy_research`（**请勿改动**上方 **SOTA** cell）

- **SOTA**：保留你原来的 `calculate_delta_threshold` / `vwap_dist_sum_v41_strategy_enhanced`，便于对照固定基线。
- **本段**：`calculate_delta_threshold_research` + `vwap_dist_sum_v41_strategy_research` — 参数可外置、`bars.copy()` 不污染原始 `bars`，兼容 pandas 2.x。
- 下方「研究工具箱」里的图与统计表，请 **`vwap_dist_sum_v41_strategy_research` + `StrategyResearchParams`** 跑完后使用。

In [18]:
# v41 研究版：与 SOTA 逻辑相同，但阈值/窗口参数可外置，且对 pandas 2.x 使用 bfill/ffill
# 在 **bars.copy()** 上运行，避免污染上游 `bars`
def calculate_delta_threshold_research(delta_series, window=100, method='mad', k=1, vol_adjusted=False, vol_series=None, local_window=10):
    if method == 'mad':
        rolling_median = delta_series.rolling(window).median()
        abs_dev = (delta_series - rolling_median).abs()
        rolling_mad = abs_dev.rolling(window).median()
        threshold = rolling_mad * k
    elif method == 'std':
        threshold = delta_series.rolling(window).std() * k
    elif method == 'quantile':
        threshold = delta_series.rolling(window).quantile(0.75)
    else:
        raise ValueError(method)

    if vol_adjusted and vol_series is not None:
        local_vol_series = vol_series.rolling(local_window).mean()
        avg_vol = local_vol_series.rolling(window).mean()
        threshold = threshold * np.sqrt(local_vol_series / (avg_vol + 1e-15))

    return threshold.bfill().ffill()


def vwap_dist_sum_v41_strategy_research(
    bars,
    fw1, ema_w1, fw2, ema_w2,
    fill_rate=0.7,
    save_metrics=False,
    plot_enhanced=False,
    *,
    thr_window=100,
    thr_method="mad",
    k_delta1=1.5,
    k_delta2=1.0,
    vol_adjusted_thr=True,
    local_vol_window=10,
):
    bars = bars.copy()
    factor_name = 'vwap_dist_sum_imbalance_v41'

    bars[f'buy_vwap_dist_sum_fw{fw1}'] = bars['buy_vwap_dist_sum'].rolling(fw1).mean()
    bars[f'sell_vwap_dist_sum_fw{fw1}'] = bars['sell_vwap_dist_sum'].rolling(fw1).mean()
    bars[f'{factor_name}_fw{fw1}'] = (bars[f'buy_vwap_dist_sum_fw{fw1}'] - bars[f'sell_vwap_dist_sum_fw{fw1}']) / (bars[f'buy_vwap_dist_sum_fw{fw1}'] + bars[f'sell_vwap_dist_sum_fw{fw1}'] + 1e-10)
    bars[f'{factor_name}_fw{fw1}_ema{ema_w1}'] = bars[f'{factor_name}_fw{fw1}'].ewm(span=ema_w1, min_periods=1).mean()

    bars[f'buy_vwap_dist_sum_fw{fw2}'] = bars['buy_vwap_dist_sum'].rolling(fw2).mean()
    bars[f'sell_vwap_dist_sum_fw{fw2}'] = bars['sell_vwap_dist_sum'].rolling(fw2).mean()
    bars[f'{factor_name}_fw{fw2}'] = (bars[f'buy_vwap_dist_sum_fw{fw2}'] - bars[f'sell_vwap_dist_sum_fw{fw2}']) / (bars[f'buy_vwap_dist_sum_fw{fw2}'] + bars[f'sell_vwap_dist_sum_fw{fw2}'] + 1e-10)
    bars[f'{factor_name}_fw{fw2}_ema{ema_w2}'] = bars[f'{factor_name}_fw{fw2}'].ewm(span=ema_w2, min_periods=1).mean()

    bars['delta1'] = bars[f'{factor_name}_fw{fw1}_ema{ema_w1}'].diff()
    bars['delta2'] = bars[f'{factor_name}_fw{fw2}_ema{ema_w2}'].diff()

    bars['delta_threshold1'] = calculate_delta_threshold_research(
        bars['delta1'], window=thr_window, method=thr_method, k=k_delta1,
        vol_adjusted=vol_adjusted_thr, vol_series=bars['volume'], local_window=local_vol_window,
    )
    bars['delta_threshold2'] = calculate_delta_threshold_research(
        bars['delta2'], window=thr_window, method=thr_method, k=k_delta2,
        vol_adjusted=vol_adjusted_thr, vol_series=bars['volume'], local_window=local_vol_window,
    )

    bars['delta_sign_raw1'] = np.sign(bars['delta1'])
    bars['delta_sign1'] = bars['delta_sign_raw1'].replace(0, np.nan).ffill()
    bars['delta_sign_raw2'] = np.sign(bars['delta2'])
    bars['delta_sign2'] = bars['delta_sign_raw2'].replace(0, np.nan).ffill()

    bars['reversal_neg'] = (bars['delta_sign2'].shift(1) == 1) & (bars['delta_sign2'] == -1)
    bars['reversal_pos'] = (bars['delta_sign1'].shift(1) == -1) & (bars['delta_sign1'] == 1)

    filter_delta1 = bars['delta1'].abs() > bars['delta_threshold1']
    filter_delta2 = bars['delta2'].abs() > bars['delta_threshold2']

    bars[f'short_entry_{factor_name}'] = bars['reversal_neg'] & filter_delta2
    bars[f'long_entry_{factor_name}'] = bars['reversal_pos'] & filter_delta1

    bars[f'position_{factor_name}'] = np.nan
    bars.loc[bars[f'short_entry_{factor_name}'], f'position_{factor_name}'] = -1
    bars.loc[bars[f'long_entry_{factor_name}'], f'position_{factor_name}'] = 1
    bars[f'position_{factor_name}'].ffill(inplace=True)
    bars[f'position_{factor_name}'] = bars[f'position_{factor_name}'].fillna(0)

    fee_rate = (2 * fill_rate + 5 * (1 - fill_rate)) * 1e-4
    bars['returns'] = (bars['close'] - bars['open']) / bars['open']
    bars[f'strategy_returns_{factor_name}'] = bars[f'position_{factor_name}'].shift(1) * bars['returns']
    bars['position_change'] = bars[f'position_{factor_name}'].diff().fillna(0)
    bars[f'strategy_returns_{factor_name}'] = bars[f'strategy_returns_{factor_name}'] - fee_rate * bars['position_change'].abs()

    bars['benchmark_cumulative'] = (1 + bars['returns']).cumprod()
    bars[f'strategy_cumulative_{factor_name}'] = (1 + bars[f'strategy_returns_{factor_name}']).cumprod()

    metrics, trades_df, bars_with_metrics = calculate_trading_metrics(
        bars, f'position_{factor_name}', f'strategy_returns_{factor_name}',
        fee_rate, factor_name, fw1, ema_w1
    )
    if save_metrics:
        output_dir = Path("/Volumes/Lexar/mean_reversion_data/backtest_results")
        save_metrics_to_file(metrics, trades_df, output_dir, factor_name, fw1, ema_w1)
    if plot_enhanced:
        output_dir = Path("/Volumes/Lexar/mean_reversion_data/backtest_results")
        plot_enhanced_results(bars_with_metrics, metrics, trades_df, factor_name, fw1, ema_w1, output_dir)

    return bars_with_metrics, metrics, trades_df

### 研究工具箱：指标 / 函数怎么用

| 对象 | 用途 |
|------|------|
| `StrategyResearchParams` | 集中记录 symbol、bar 规格、fw/ema、**动态阈值**（窗口/方法/k/是否 volume 调节）、手续费假设。敏感性：`from dataclasses import replace` → `replace(RESEARCH_PARAMS, k_delta1=2.0)` |
| `apply_cta_matplotlib_style()` | 统一白底、网格、字号、色板；报告出图前调用一次即可 |
| `plot_v41_signal_panel(bars, p, slice_last_n=...)` | 价格+双因子、Δ1/Δ2 与 ±thr、仓位与入场点 |
| `summarize_signal_statistics(bars, p)` | 分布分位数、阈值穿越比例、|Δ|>thr 时下一根 bar 收益均值（粗看阈值经济含义） |
| `threshold_sensitivity_table(bars, p, k_grid=...)` | 扫描 k → 穿越率，用于定 k 量级 |
| `plot_delta_distribution_with_threshold` | Δ 与 |thr| 的直方图 |
| `compare_bar_types_factor_stability` | 从 **merged trades** 出发对比 tpb vs volume bar 的因子 std |

In [24]:
from dataclasses import dataclass, replace
from typing import Iterable, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from cycler import cycler as mpl_cycler

try:
    import seaborn as sns
    _HAS_SNS = True
except ImportError:
    _HAS_SNS = False


@dataclass
class StrategyResearchParams:
    symbol: str = "SOL/USDC"
    bar_mode: str = "trade_count"
    trades_per_bar: int = 500
    volume_per_bar: float = 0.0
    fw1: int = 8
    ema_w1: int = 10
    fw2: int = 10
    ema_w2: int = 10
    thr_window: int = 100
    thr_method: str = "mad"
    k_delta1: float = 1.5
    k_delta2: float = 1.0
    vol_adjusted_thr: bool = True
    local_vol_window: int = 10
    fill_rate: float = 0.7
    fee_maker_bps: float = 2.0
    fee_taker_bps: float = 5.0

    def fee_rate(self) -> float:
        return (self.fee_maker_bps * self.fill_rate + self.fee_taker_bps * (1.0 - self.fill_rate)) * 1e-4

    def describe_bar_line(self) -> str:
        if self.bar_mode == "volume" and self.volume_per_bar > 0:
            return f"merged+volume (target_vol={self.volume_per_bar:g})"
        return f"merged+tpb={self.trades_per_bar}"

In [26]:
def apply_cta_matplotlib_style() -> None:
    plt.rcParams.update({
        "figure.figsize": (12, 8),
        "figure.dpi": 100,
        "savefig.dpi": 150,
        "savefig.bbox": "tight",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "grid.linestyle": "--",
        "legend.fontsize": 9,
        "axes.prop_cycle": mpl_cycler(
            color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
        ),
    })
    if _HAS_SNS:
        sns.set_theme(style="whitegrid", context="notebook")


def _v41_column_names(p: StrategyResearchParams) -> Tuple[str, str, str]:
    fn = "vwap_dist_sum_imbalance_v41"
    c1 = f"{fn}_fw{p.fw1}_ema{p.ema_w1}"
    c2 = f"{fn}_fw{p.fw2}_ema{p.ema_w2}"
    return fn, c1, c2

In [ ]:
def plot_v41_signal_panel(
    bars: pd.DataFrame,
    p: StrategyResearchParams,
    *,
    slice_last_n: Optional[int] = 2000,
    show_entries: bool = True,
    title_suffix: str = "",
) -> plt.Figure:
    apply_cta_matplotlib_style()
    fn, c1, c2 = _v41_column_names(p)
    pos_col = f"position_{fn}"
    need = {"close", "delta1", "delta2", "delta_threshold1", "delta_threshold2", c1, c2, pos_col}
    missing = need - set(bars.columns)
    if missing:
        raise ValueError(f"bars 缺少列: {sorted(missing)}")

    dfb = bars.iloc[-slice_last_n:] if slice_last_n else bars
    idx = dfb.index

    fig, axes = plt.subplots(
        4, 1, sharex=True, figsize=(14, 11),
        gridspec_kw={"height_ratios": [1.15, 1.0, 1.0, 0.65], "hspace": 0.08},
    )
    period = f"{idx[0]} — {idx[-1]}"
    fig.suptitle(
        f"{p.symbol} | {p.describe_bar_line()} | {period}\n"
        f"v41 research: fw1={p.fw1} ema1={p.ema_w1} | fw2={p.fw2} ema2={p.ema_w2} | "
        f"thr(w={p.thr_window}, {p.thr_method}, k1={p.k_delta1}, k2={p.k_delta2})"
        + (f" | {title_suffix}" if title_suffix else ""),
        fontsize=11, y=1.02,
    )

    ax = axes[0]
    ax.plot(idx, dfb["close"], color="#1f77b4", lw=0.9, label="Close")
    ax.set_ylabel("Price")
    ax.legend(loc="upper left")
    axr = ax.twinx()
    axr.plot(idx, dfb[c1], color="#ff7f0e", lw=0.75, alpha=0.9, label=f"F1 (fw{p.fw1})")
    axr.plot(idx, dfb[c2], color="#2ca02c", lw=0.75, alpha=0.9, label=f"F2 (fw{p.fw2})")
    axr.set_ylabel("Factor")
    axr.legend(loc="upper right")

    for ax_i, dcol, tcol, name in [
        (1, "delta1", "delta_threshold1", "Δ1"),
        (2, "delta2", "delta_threshold2", "Δ2"),
    ]:
        ax = axes[ax_i]
        thr = dfb[tcol].abs()
        ax.fill_between(idx, -thr, thr, color="0.75", alpha=0.25, label="±thr")
        ax.plot(idx, dfb[dcol], color="0.15", lw=0.7, label=dcol)
        ax.axhline(0, color="0.5", lw=0.6, ls=":")
        ax.set_ylabel(name)
        ax.legend(loc="upper left", ncol=2)

    ax = axes[3]
    ax.step(idx, dfb[pos_col], where="post", color="#444444", lw=0.9, label="Position")
    ax.set_ylabel("Pos")
    ax.set_ylim(-1.25, 1.25)
    ax.legend(loc="upper left")

    if show_entries:
        le, se = f"long_entry_{fn}", f"short_entry_{fn}"
        if le in dfb.columns:
            m = dfb[dfb[le]]
            ax.scatter(m.index, m[pos_col], marker="^", s=36, c="#2ca02c", zorder=5, label="long")
        if se in dfb.columns:
            m = dfb[dfb[se]]
            ax.scatter(m.index, m[pos_col], marker="v", s=36, c="#d62728", zorder=5, label="short")

    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axes[-1].xaxis.get_major_locator()))
    axes[-1].set_xlabel("Time (UTC)")
    fig.align_ylabels(axes)
    return fig

In [ ]:
def summarize_signal_statistics(bars: pd.DataFrame, p: StrategyResearchParams) -> pd.DataFrame:
    fn, c1, c2 = _v41_column_names(p)
    r = bars["returns"] if "returns" in bars.columns else (bars["close"] - bars["open"]) / bars["open"]
    fwd1 = r.shift(-1)

    def _row(name, x, thr=None):
        x = x.dropna()
        q = x.quantile([0.01, 0.05, 0.5, 0.95, 0.99])
        out = {
            "series": name, "n": len(x), "mean": x.mean(), "std": x.std(),
            "skew": stats.skew(x, nan_policy="omit"),
            "kurtosis": stats.kurtosis(x, nan_policy="omit"),
            "p01": q.loc[0.01], "p05": q.loc[0.05], "p50": q.loc[0.5],
            "p95": q.loc[0.95], "p99": q.loc[0.99],
        }
        if thr is not None:
            m = thr.reindex(x.index).dropna()
            ratio = (x.abs() / (m + 1e-15)).dropna()
            hit = (x.abs() > m).astype(float)
            out["median_abs_delta_over_thr"] = float(ratio.median())
            out["frac_bar_thr_cross"] = float(hit.mean())
            sub = fwd1.reindex(hit.index).dropna()
            hi = hit.loc[sub.index] == 1
            lo = hit.loc[sub.index] == 0
            if hi.any():
                out["fwd_ret_mean_when_|d|>thr"] = float(sub[hi].mean())
            if lo.any():
                out["fwd_ret_mean_when_|d|<=thr"] = float(sub[lo].mean())
        return out

    rows = [
        _row("factor1_smooth", bars[c1]),
        _row("factor2_smooth", bars[c2]),
        _row("delta1", bars["delta1"], bars["delta_threshold1"]),
        _row("delta2", bars["delta2"], bars["delta_threshold2"]),
    ]
    return pd.DataFrame(rows)


def threshold_sensitivity_table(
    bars: pd.DataFrame,
    p: StrategyResearchParams,
    k_grid: Iterable[float] = (0.8, 1.0, 1.2, 1.5, 2.0, 2.5),
) -> pd.DataFrame:
    rows = []
    for k in k_grid:
        t1 = calculate_delta_threshold_research(
            bars["delta1"], window=p.thr_window, method=p.thr_method, k=k,
            vol_adjusted=p.vol_adjusted_thr, vol_series=bars["volume"], local_window=p.local_vol_window,
        )
        cross = (bars["delta1"].abs() > t1).mean()
        rows.append({"k": k, "frac_|d1|>thr": float(cross), "median_thr": float(t1.median())})
    return pd.DataFrame(rows)

In [ ]:
def plot_delta_distribution_with_threshold(
    bars: pd.DataFrame,
    p: StrategyResearchParams,
    which: str = "delta1",
) -> plt.Figure:
    apply_cta_matplotlib_style()
    d = bars[which].dropna()
    thr_col = "delta_threshold1" if which == "delta1" else "delta_threshold2"
    thr = bars[thr_col].reindex(d.index).dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(d, bins=80, density=True, color="#4c72b0", alpha=0.75, edgecolor="white")
    axes[0].axvline(0, color="0.4", ls="--", lw=0.8)
    axes[0].set_title(f"{which} histogram")
    axes[0].set_xlabel(which)
    axes[1].hist(thr, bins=60, density=True, color="#dd8452", alpha=0.75, edgecolor="white")
    axes[1].set_title(f"|{thr_col}| histogram")
    fig.suptitle(f"{p.symbol} | thr_window={p.thr_window} {p.thr_method}")
    fig.tight_layout()
    return fig

In [39]:
RESEARCH_PARAMS = StrategyResearchParams(
    trades_per_bar=trades_per_bar,
    fw1=8, ema_w1=10, fw2=10, ema_w2=10,
    thr_window=100, thr_method="mad", k_delta1=1.5, k_delta2=1.0,
    vol_adjusted_thr=True, local_vol_window=10,
    fill_rate=0.7,
)

In [ ]:
bars_research, metrics, trades_df = vwap_dist_sum_v41_strategy_research(
    bars,
    RESEARCH_PARAMS.fw1,
    RESEARCH_PARAMS.ema_w1,
    RESEARCH_PARAMS.fw2,
    RESEARCH_PARAMS.ema_w2,
    fill_rate=RESEARCH_PARAMS.fill_rate,
    save_metrics=False,
    plot_enhanced=False,
    thr_window=RESEARCH_PARAMS.thr_window,
    thr_method=RESEARCH_PARAMS.thr_method,
    k_delta1=RESEARCH_PARAMS.k_delta1,
    k_delta2=RESEARCH_PARAMS.k_delta2,
    vol_adjusted_thr=RESEARCH_PARAMS.vol_adjusted_thr,
    local_vol_window=RESEARCH_PARAMS.local_vol_window,
)

In [ ]:
apply_cta_matplotlib_style()
fig_panel = plot_v41_signal_panel(bars_research, RESEARCH_PARAMS, slice_last_n=2500)
plt.tight_layout()
plt.show()

display(summarize_signal_statistics(bars_research, RESEARCH_PARAMS))
display(threshold_sensitivity_table(bars_research, RESEARCH_PARAMS))

fig_hist = plot_delta_distribution_with_threshold(bars_research, RESEARCH_PARAMS, which="delta1")
plt.show()

#### Rolling IC（预测力是否随时间漂移）

- **输入**：`bars_research`（已含 `returns` 与平滑因子列）。
- **含义**：滚动 Pearson 相关(因子, 下一根bar收益)。窗口越大曲线越平滑；仅作探索，**非**样本外检验。
- **用法**：若 IC 长期贴 0 或符号翻转频繁 → 考虑分 regime 重估 `fw/ema/thr_window`，或缩短持有/换因子。

In [ ]:
def plot_rolling_factor_ic(
    bars: pd.DataFrame,
    p: StrategyResearchParams,
    *,
    window: int = 800,
    min_periods: int = 200,
) -> plt.Figure:
    _, c1, c2 = _v41_column_names(p)
    r = bars["returns"] if "returns" in bars.columns else (bars["close"] - bars["open"]) / bars["open"]
    fwd = r.shift(-1)
    ic1 = bars[c1].rolling(window, min_periods=min_periods).corr(fwd)
    ic2 = bars[c2].rolling(window, min_periods=min_periods).corr(fwd)
    apply_cta_matplotlib_style()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(bars.index, ic1, lw=0.85, label=f"IC F1 (w={window})")
    ax.plot(bars.index, ic2, lw=0.85, label=f"IC F2 (w={window})")
    ax.axhline(0, color="0.45", ls=":", lw=0.8)
    ax.set_ylabel("Correlation")
    ax.set_title(f"{p.symbol} | rolling IC (exploratory)")
    ax.legend(loc="upper left")
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.tight_layout()
    return fig

In [ ]:
fig_ic = plot_rolling_factor_ic(bars_research, RESEARCH_PARAMS, window=800, min_periods=200)
plt.show()

## 对比使用 Volume 进行聚合

#### 流程（从原始数据出发）

1. **输入**：上方已得到的 `trades_df`（merged taker）、以及当前 notebook 的 `trades_per_bar`。
2. **问题**：同样因子在 **trade-count bar** vs **volume bar** 下，波动率（std）与 bar 时长分布是否更稳定？
3. **输出**：表 `bar_cmp` + 柱状图；若 volume bar 的 factor std 明显更低且 bar 时长更可控，可优先试 volume 聚合。

In [19]:
# Step 1 — 仅依赖 merged trades，不重新读盘
assert "volume" in trades_df.columns

### 比较 volume vs. trade count 聚合的因子稳定性

从 merged trades 的 vol median 出发，bar 的 trade count 定为 50, 100, 200, 500...
对应的，把 bar 的 volume 设置成 vol median 的相应 multiple

In [ ]:
def compare_bar_types_factor_stability(
    merged_trades: pd.DataFrame,
    *,
    trades_per_bar: int = 200,
    volume_per_bar_candidates: Optional[list] = None,
    factor_name: str = "vwap_dist_sum_imbalance",
    factor_window: int = 10,
) -> pd.DataFrame:
    from rule_backtest.data_loader import build_trade_count_bars_merged, build_volume_bars_merged

    b_tc = build_trade_count_bars_merged(merged_trades, trades_per_bar)
    f_tc = compute_factor(factor_name, b_tc, window=factor_window)

    med_v = float(merged_trades["volume"].median()) # 这里用 median 过小，导致 volume bar 的 std 偏大
    if volume_per_bar_candidates is None:
        volume_per_bar_candidates = sorted({med_v * 50, med_v * 100, med_v * 200, med_v * 500})

    rows = [{
        "bar_type": "trade_count",
        "param": trades_per_bar,
        "n_bars": len(b_tc),
        "bar_dur_sec_med": b_tc["bar_duration_sec"].median() if "bar_duration_sec" in b_tc.columns else np.nan,
        "factor_std": float(f_tc.std()),
        "factor_abs_mean": float(f_tc.abs().mean()),
    }]
    for vpb in volume_per_bar_candidates:
        if vpb <= 0:
            continue
        b_v = build_volume_bars_merged(merged_trades, float(vpb))
        if b_v.empty:
            continue
        f_v = compute_factor(factor_name, b_v, window=factor_window)
        rows.append({
            "bar_type": "volume",
            "param": vpb,
            "n_bars": len(b_v),
            "bar_dur_sec_med": b_v["bar_duration_sec"].median() if "bar_duration_sec" in b_v.columns else np.nan,
            "factor_std": float(f_v.std()),
            "factor_abs_mean": float(f_v.abs().mean()),
        })
    return pd.DataFrame(rows)

In [32]:
trades_df['volume'].describe()

count    3.032559e+06
mean     4.792637e+01
std      1.598329e+02
min      1.000000e-02
25%      1.030000e+00
50%      9.370000e+00
75%      3.932000e+01
max      2.325747e+04
Name: volume, dtype: float64

#### volume 很典型的厚尾分布，不能用 trades volume 的 median, 这个 median 一定偏小

In [35]:
from rule_backtest.data_loader import build_trade_count_bars_merged, build_volume_bars_merged
trade_cnt_bars_200 =build_trade_count_bars_merged(trades_df, 200)
trade_cnt_bars_200['volume'].describe()

count    15163.000000
mean      9585.144844
std       4549.421640
min       1185.200000
25%       6471.780000
50%       8558.550000
75%      11594.310000
max      47706.610000
Name: volume, dtype: float64

#### 我们需要在 bar time seconds 差不多的情况下 factor std 尽量小

In [36]:
# Step 2 — 对比表（可改 factor_window / volume_per_bar_candidates）
bar_cmp_fw10 = compare_bar_types_factor_stability(
    trades_df,
    trades_per_bar=trades_per_bar,
    factor_name="vwap_dist_sum_imbalance",
    factor_window=10,
    volume_per_bar_candidates=[5000, 10000, 20000, 50000]
)
display(bar_cmp_fw10)

,bar_type,param,n_bars,bar_dur_sec_med,factor_std,factor_abs_mean
0,trade_count,200,15163,127.3960,0.135219,0.109717
1,volume,5000,27549,59.2850,0.168850,0.135713
2,volume,10000,14134,124.2110,0.138931,0.111637
3,volume,20000,7166,259.2745,0.111443,0.089089
4,volume,50000,2891,701.3790,0.079515,0.062779


In [37]:
bar_cmp_fw8 = compare_bar_types_factor_stability(
    trades_df,
    trades_per_bar=trades_per_bar,
    factor_name="vwap_dist_sum_imbalance",
    factor_window=8,
    volume_per_bar_candidates=[5000, 10000, 20000, 50000]
)
display(bar_cmp_fw8)

,bar_type,param,n_bars,bar_dur_sec_med,factor_std,factor_abs_mean
0,trade_count,200,15163,127.3960,0.143957,0.116770
1,volume,5000,27549,59.2850,0.180406,0.144867
2,volume,10000,14134,124.2110,0.148510,0.119352
3,volume,20000,7166,259.2745,0.120306,0.096237
4,volume,50000,2891,701.3790,0.086661,0.068576


In [ ]:
# Step 3 — 因子波动可视化（与 Step 2 首张表对齐；若只用 fw8 可改为 bar_cmp_fw8）
bar_cmp = bar_cmp_fw10
fig, ax = plt.subplots(figsize=(8, 4))
apply_cta_matplotlib_style()
x = np.arange(len(bar_cmp))
ax.bar(x, bar_cmp["factor_std"].values, color=["#4c72b0"] + ["#dd8452"] * (len(bar_cmp) - 1))
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{a}: {b}" for a, b in zip(bar_cmp["bar_type"], bar_cmp["param"])],
    fontsize=8,
)
ax.set_ylabel("Factor std")
ax.set_title("Bar spec vs factor volatility")
fig.tight_layout()
plt.show()

### 使用和 tpb = 200 相匹配的 volume 参数进行 v41 的测试

- **含义**：在 **merged** 成交上，先建 **`tpb = 200`** 的 trade-count bar，取其每根 bar 的 **`volume` 中位数** 作为 volume bar 目标量 **`VPB_MATCH`**，使 volume bar 的时长与 tpb=200 同量级（与上方对比表里 **vpb ≈ 10000** 附近、`bar_dur_sec_med` 接近 **127s** 的档一致）。
- **做法**：`build_volume_bars_merged(trades_df, VPB_MATCH)`，再在同一套 `RESEARCH_PARAMS`（fw / ema / k / thr）下调用 `vwap_dist_sum_v41_strategy_research`。
- **依赖**：已运行定义 `RESEARCH_PARAMS`、`vwap_dist_sum_v41_strategy_research`、`replace`、`print_metrics_summary` 的 cell；`trades_df` 为 merged 成交表。

In [40]:
# 与 tpb=200 每根 bar 实际成交量中位数对齐（不依赖单笔成交 volume 的 median）


VPB_MATCH = float(trade_cnt_bars_200["volume"].median())
bars_v41_vol_match = build_volume_bars_merged(trades_df, VPB_MATCH)

RESEARCH_PARAMS_VOL_MATCH = replace(
    RESEARCH_PARAMS,
    bar_mode="volume",
    volume_per_bar=VPB_MATCH,
    trades_per_bar=trades_per_bar,
)

bars_v41_vol_match, metrics_v41_vol_match, v41_trades_vol_match = vwap_dist_sum_v41_strategy_research(
    bars_v41_vol_match,
    RESEARCH_PARAMS_VOL_MATCH.fw1,
    RESEARCH_PARAMS_VOL_MATCH.ema_w1,
    RESEARCH_PARAMS_VOL_MATCH.fw2,
    RESEARCH_PARAMS_VOL_MATCH.ema_w2,
    fill_rate=RESEARCH_PARAMS_VOL_MATCH.fill_rate,
    save_metrics=False,
    plot_enhanced=False,
    thr_window=RESEARCH_PARAMS_VOL_MATCH.thr_window,
    thr_method=RESEARCH_PARAMS_VOL_MATCH.thr_method,
    k_delta1=RESEARCH_PARAMS_VOL_MATCH.k_delta1,
    k_delta2=RESEARCH_PARAMS_VOL_MATCH.k_delta2,
    vol_adjusted_thr=RESEARCH_PARAMS_VOL_MATCH.vol_adjusted_thr,
    local_vol_window=RESEARCH_PARAMS_VOL_MATCH.local_vol_window,
)

print(f"VPB_MATCH (median bar volume @ merged tpb={trades_per_bar}) = {VPB_MATCH:,.4f}")
print(f"n_bars={len(bars_v41_vol_match)} | {RESEARCH_PARAMS_VOL_MATCH.describe_bar_line()}")
print_metrics_summary(metrics_v41_vol_match)

VPB_MATCH (median bar volume @ merged tpb=200) = 8,558.5500
n_bars=16438 | merged+volume (target_vol=8558.55)
策略绩效报告 - vwap_dist_sum_imbalance_v41 (fw=8, ema=10)
回测期间: 2026-02-02 16:00:01.690000 至 2026-03-04 15:59:58.694000
总Bar数: 16438
------------------------------------------------------------
【收益指标】
  策略总收益 (净): -41.55%
  多头收益 (原始): -23.58%
  空头收益 (原始): -19.47%
  基准收益: -10.66%
  年化收益率: -100.00%
------------------------------------------------------------
【交易统计】
  总交易次数: 89
  多头交易: 44
  空头交易: 45
  反向交易: 88
  平均持仓Bar数: 184.10
------------------------------------------------------------
【胜率统计】
  总胜率: 47.19%
  多头胜率: 43.18%
  空头胜率: 51.11%
------------------------------------------------------------
【盈亏分析】
  盈亏比 (Profit Factor): 0.622
  平均盈利: 1.7421%
  平均亏损: -2.5034%
  单笔期望收益: -0.4999%
  最佳交易: 7.8136%
  最差交易: -12.5548%
------------------------------------------------------------
【风险指标】
  夏普比率: -1.740
  最大回撤: -44.04%
  最大回撤持续期 (bars): 52
  卡尔玛比率: -0.943
  年化波动率: 161.28%
------------------

#### 补充：merged + **vpb = 10000**（固定档）

与上方 `bar_cmp_*` 中 **param = 10000** 一致：`bar_dur_sec_med` 与 **tpb = 200** 接近，便于与 `VPB_MATCH`（中位数对齐）结果并排对照。

In [41]:
from rule_backtest.data_loader import build_volume_bars_merged

VPB_FIXED = 10_000.0
bars_v41_vpb10k = build_volume_bars_merged(trades_df, VPB_FIXED)

RESEARCH_PARAMS_VPB10K = replace(
    RESEARCH_PARAMS,
    bar_mode="volume",
    volume_per_bar=VPB_FIXED,
    trades_per_bar=trades_per_bar,
)

bars_v41_vpb10k, metrics_v41_vpb10k, v41_trades_vpb10k = vwap_dist_sum_v41_strategy_research(
    bars_v41_vpb10k,
    RESEARCH_PARAMS_VPB10K.fw1,
    RESEARCH_PARAMS_VPB10K.ema_w1,
    RESEARCH_PARAMS_VPB10K.fw2,
    RESEARCH_PARAMS_VPB10K.ema_w2,
    fill_rate=RESEARCH_PARAMS_VPB10K.fill_rate,
    save_metrics=False,
    plot_enhanced=False,
    thr_window=RESEARCH_PARAMS_VPB10K.thr_window,
    thr_method=RESEARCH_PARAMS_VPB10K.thr_method,
    k_delta1=RESEARCH_PARAMS_VPB10K.k_delta1,
    k_delta2=RESEARCH_PARAMS_VPB10K.k_delta2,
    vol_adjusted_thr=RESEARCH_PARAMS_VPB10K.vol_adjusted_thr,
    local_vol_window=RESEARCH_PARAMS_VPB10K.local_vol_window,
)

print(f"VPB_FIXED = {VPB_FIXED:,.0f} | n_bars={len(bars_v41_vpb10k)} | {RESEARCH_PARAMS_VPB10K.describe_bar_line()}")
print_metrics_summary(metrics_v41_vpb10k)

VPB_FIXED = 10,000 | n_bars=14134 | merged+volume (target_vol=10000)
策略绩效报告 - vwap_dist_sum_imbalance_v41 (fw=8, ema=10)
回测期间: 2026-02-02 16:00:01.690000 至 2026-03-04 15:59:31.925000
总Bar数: 14134
------------------------------------------------------------
【收益指标】
  策略总收益 (净): -12.99%
  多头收益 (原始): -7.03%
  空头收益 (原始): -3.17%
  基准收益: -11.29%
  年化收益率: -99.43%
------------------------------------------------------------
【交易统计】
  总交易次数: 59
  多头交易: 29
  空头交易: 30
  反向交易: 58
  平均持仓Bar数: 239.20
------------------------------------------------------------
【胜率统计】
  总胜率: 50.85%
  多头胜率: 44.83%
  空头胜率: 56.67%
------------------------------------------------------------
【盈亏分析】
  盈亏比 (Profit Factor): 0.909
  平均盈利: 2.1496%
  平均亏损: -2.4475%
  单笔期望收益: -0.1100%
  最佳交易: 9.6490%
  最差交易: -12.8226%
------------------------------------------------------------
【风险指标】
  夏普比率: -0.348
  最大回撤: -34.12%
  最大回撤持续期 (bars): 20
  卡尔玛比率: -0.381
  年化波动率: 173.22%
------------------------------------------------------------
【

## 对比 sqrt(volume) 调整后的信号

但根据 volume 聚合应该是从根本上解决问题

#### 流程：sqrt(volume) 缩放（bar 内 volume 仍用 time/tpb 时）

- **动机**：bar 内成交量越大，delta 噪声往往越大；用 $\sqrt{V}$ 或 $V^\alpha$ 做尺度归一属于**临时**稳定化，**volume bar** 才是结构层解决。
- **步骤**：在现有 `bars` 上对 `delta1/delta2` 除以 `np.sqrt(bar_volume.clip(lower=eps))`，再重新算阈值与统计；与原始序列对照分布。

In [ ]:
# 依赖 bars_research（含 delta1/2 与 bar volume）
eps = 1e-9
vol_scale = np.sqrt(bars_research["volume"].clip(lower=eps))
delta1_s = bars_research["delta1"] / vol_scale
delta2_s = bars_research["delta2"] / vol_scale

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
apply_cta_matplotlib_style()
ax[0].hist(bars_research["delta1"].dropna(), bins=60, density=True, alpha=0.6, label="delta1 raw")
ax[0].hist(delta1_s.dropna(), bins=60, density=True, alpha=0.6, label="delta1 / sqrt(V)")
ax[0].legend()
ax[0].set_title("Delta1 distribution")
ax[1].hist(bars_research["delta2"].dropna(), bins=60, density=True, alpha=0.6, label="delta2 raw")
ax[1].hist(delta2_s.dropna(), bins=60, density=True, alpha=0.6, label="delta2 / sqrt(V)")
ax[1].legend()
ax[1].set_title("Delta2 distribution")
fig.suptitle("sqrt(volume) scaling on bars_research")
fig.tight_layout()
plt.show()

## 基于 +DI, -DI 对开仓阈值进行调整
对 k_delta1 和 k_delta2 进行根据趋势的动态调整，
确保趋势环境下，更易开顺势仓位，难开逆势仓位

#### 流程：用 +DI / -DI（Wilder）调节阈值倍数

- **数据**：仅用 bar 的 OHLC（与 CTA 常用一致）。
- **想法**：趋势强（ADX 高且 +DI>-DI）时 **提高** 逆势反转类信号的阈值（`k` 乘子 >1），震荡时恢复基准。
- **下步**：将 `k_eff = k * mult` 代入 `calculate_delta_threshold_research`（需在本节复制一行阈值计算做对照实验）。

In [ ]:
def wilder_adx_di(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    h, l, c = df["high"], df["low"], df["close"]
    prev_c = c.shift(1)
    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    up = h.diff()
    down = -l.diff()
    plus_dm = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)
    atr = tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1 / period, min_periods=period, adjust=False).mean() / (atr + 1e-15)
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1 / period, min_periods=period, adjust=False).mean() / (atr + 1e-15)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-15)
    adx = dx.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    out = df.copy()
    out["plus_di"], out["minus_di"], out["adx"] = plus_di, minus_di, adx
    return out


bars_adx = wilder_adx_di(bars_research, period=14)

In [ ]:
# 示例：趋势状态下抬高 delta1 阈值倍数（仅演示 mult 构造，不覆盖 bars_research）
trend_long = (bars_adx["plus_di"] > bars_adx["minus_di"]) & (bars_adx["adx"] > 25)
mult = np.where(trend_long, 1.25, 1.0)
pd.Series(mult, index=bars_adx.index).value_counts().head()

## 使用 MAE, MFE 进行止损优化

#### 流程：MAE / MFE（入场后 N 根 bar 内极值）

- **定义**：MAE = 对持仓不利的最坏变动；MFE = 有利方向最大变动（均以入场价/开盘价为锚）。
- **用途**：统计分布 → 设定**初始止损**与**止盈回撤**的实证分位数；再回测扫描 N。

In [ ]:
def per_trade_mae_mfe(ohlc: pd.DataFrame, pos: pd.Series, max_hold: int = 30):
    pos = pos.reindex(ohlc.index).fillna(0)
    entries = pos.diff().fillna(0).abs() > 0
    # simplified: use long-only excursion for demo
    records = []
    i = 0
    n = len(ohlc)
    while i < n:
        if not entries.iloc[i] or pos.iloc[i] == 0:
            i += 1
            continue
        direction = np.sign(pos.iloc[i])
        entry_px = float(ohlc["open"].iloc[i])
        end = min(n, i + max_hold)
        window = ohlc.iloc[i:end]
        if direction > 0:
            mfe = (window["high"].max() / entry_px - 1) * 1e4
            mae = (window["low"].min() / entry_px - 1) * 1e4
        else:
            mfe = (1 - window["low"].min() / entry_px) * 1e4
            mae = (1 - window["high"].max() / entry_px) * 1e4
        records.append({"t": ohlc.index[i], "mfe_bps": mfe, "mae_bps": mae})
        i += 1
    return pd.DataFrame(records)


fn = "vwap_dist_sum_imbalance_v41"
pos_s = bars_research[f"position_{fn}"]
mae_tbl = per_trade_mae_mfe(bars_research[["open", "high", "low", "close"]], pos_s, max_hold=40)
display(mae_tbl.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

In [ ]:
if not mae_tbl.empty:
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    apply_cta_matplotlib_style()
    ax[0].hist(mae_tbl["mfe_bps"], bins=40, color="#2ca02c", alpha=0.75)
    ax[0].set_title("MFE (bps)")
    ax[1].hist(mae_tbl["mae_bps"], bins=40, color="#d62728", alpha=0.75)
    ax[1].set_title("MAE (bps)")
    fig.tight_layout()
    plt.show()

## 尝试 Grid Trading

#### 流程：网格交易（研究草图）

- **数据**：`bars_research` 的 `close` 与波动（如 ATR 或 rolling std）。
- **步骤**：设定网格间距 $\Delta$、层数、是否对称；用向量化回测统计成交次数与库存偏斜；**与信号策略正交**，单独评估。

In [ ]:
# 占位：网格参数与简易 PnL 需按你的风控再接；此处仅保留价格路径与网格线可视化
px = bars_research["close"].iloc[-500:]
grid_step = float(px.pct_change().std() * px.mean() * 1.5)  # heuristic
center = float(px.median())
levels = [center + k * grid_step for k in range(-3, 4)]
fig, ax = plt.subplots(figsize=(11, 3))
apply_cta_matplotlib_style()
ax.plot(px.index, px.values, lw=0.8)
for lv in levels:
    ax.axhline(lv, color="0.6", ls=":", lw=0.6)
ax.set_title("Close vs heuristic grid levels (research placeholder)")
fig.tight_layout()
plt.show()

## 结合 Orderbook 判断 Liquidity 条件是否具备开仓条件

#### 流程：Order book / 流动性过滤

- **数据需求**：L2 快照或 spread + depth（本 notebook 仅有成交-derived bar，**无**内置 orderbook）。
- **建议**：接入存储的 bid/ask、top5 深度；定义 `spread_bps < s_max` 且 `depth > d_min` 时才允许开仓；与 `bars_research` 按时间戳 merge。

In [ ]:
# 数据未接入时仅占位
LIQUIDITY_FILTER_PLACEHOLDER = {
    "spread_bps_max": 5.0,
    "depth_usd_min": 50_000,
    "note": "Merge L2 features on timestamp before entry signal",
}
LIQUIDITY_FILTER_PLACEHOLDER

## 提高杠杆

#### 流程：杠杆（研究视角）

- **不等于**提高 alpha；只放大波动与保证金风险。
- **步骤**：固定策略净值曲线，乘以杠杆 L，扣除**预估强平距离**；用 `max_drawdown * L` 做压力情景表。

In [ ]:
L = 2.0  # 示例
eq = bars_research["strategy_cumulative_vwap_dist_sum_imbalance_v41"]
lev_eq = (1 + (eq.pct_change().fillna(0) * L)).cumprod()
fig, ax = plt.subplots(figsize=(10, 3))
apply_cta_matplotlib_style()
ax.plot(eq.index, eq.values / eq.iloc[0], label=f"equity L=1")
ax.plot(lev_eq.index, lev_eq.values / lev_eq.iloc[0], label=f"synthetic L={L}")
ax.legend()
ax.set_title("Leverage scaling (synthetic, not funding-adjusted)")
fig.tight_layout()
plt.show()

## 优化仓位配置

#### 流程：仓位配置

- **目标**：在波动与回撤约束下分配风险预算。
- **示例**：波动率倒数加权 `vol_target`、或 Kelly 上界裁剪；输入 `bars_research["returns"]` 的滚动波动与策略信号。

In [ ]:
ret = bars_research["returns"]
vol_roll = ret.rolling(100, min_periods=20).std().clip(lower=1e-6)
vol_target = 0.002  # 每 bar 目标波动，需按 bar 时长年化换算
notional_scale = (vol_target / vol_roll).clip(upper=3.0)
fig, ax = plt.subplots(figsize=(10, 2.8))
apply_cta_matplotlib_style()
ax.plot(notional_scale.index, notional_scale.values, lw=0.7)
ax.set_title("Illustrative vol-target scaling (clip max=3)")
fig.tight_layout()
plt.show()